# PCA — Exercises

Companion to the note [PCA](PCA.md).

Practise principal component analysis: the covariance / eigen view, the SVD view, projection and reconstruction, explained variance and choosing $k$, why to centre and standardize, whitening, power iteration and probabilistic PCA, first by hand and then from scratch in NumPy, checked against scikit-learn.

**16 exercises** · 🧠 Concept ×3 · ✍️ By hand ×5 · 💻 Code ×8

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from sklearn.decomposition import PCA
from sklearn.datasets import load_digits, load_wine
from sklearn.preprocessing import StandardScaler
rng = np.random.default_rng(0)
# Shared correlated 3-D data (rows = samples), centred below where needed
A_mix = np.array([[3.0, 0.0, 0.0], [1.0, 1.0, 0.0], [0.5, 0.2, 0.3]])
X_c3 = rng.normal(size=(500, 3)) @ A_mix
X_digits = load_digits().data

## Part A · Concepts

### Exercise 1 · Centre, and usually standardize
*🧠 Concept · ★☆☆*

Step 1 of the algorithm in the note is "centre the data (and usually standardize)".
(a) What goes wrong if you skip centring? (b) When should you standardize and when not?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

$S=\frac1N X^\top X$ is only the covariance if the columns of $X$ have mean zero. Variance depends on the unit of measurement.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) Without centring, $\frac1N X^\top X$ is the matrix of second moments, not the covariance. The first "component" then mostly points from the origin to the data mean, which says nothing about the spread.
(b) PCA maximizes variance, and variance depends on units: a feature in millimetres gets $10^6$ times the variance of the same feature in metres. Standardize when features have different units or scales (PCA on the correlation matrix).
Don't when all features share a meaningful common scale (e.g. pixel intensities): standardizing would inflate almost-constant noisy pixels.

</details>

### Exercise 2 · Two views of the same objective
*🧠 Concept · ★★☆*

The note says "maximize projected variance = minimize reconstruction error". For a single unit vector $w$ and centred data,
show that $\frac1N\sum_i\lVert x_i - (w^\top x_i)w\rVert^2 = \frac1N\sum_i\lVert x_i\rVert^2 - w^\top S w$, and conclude.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Pythagoras: $x_i - (w^\top x_i)w$ is orthogonal to $(w^\top x_i)w$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Since $\lVert w\rVert=1$, $x_i=(w^\top x_i)w + r_i$ with $r_i\perp w$, so $\lVert r_i\rVert^2=\lVert x_i\rVert^2-(w^\top x_i)^2$.
Averaging: $\frac1N\sum\lVert r_i\rVert^2=\frac1N\sum\lVert x_i\rVert^2-\frac1N\sum (w^\top x_i)^2=\text{const}-w^\top S w$.
The first term does not depend on $w$, so minimizing reconstruction error is the same as maximizing the projected variance $w^\top S w$. The same argument works for $k$ orthonormal directions.

</details>

### Exercise 3 · When PCA is the wrong tool
*🧠 Concept · ★☆☆*

Give a concrete example for each limitation in the note: (a) linear only, (b) sensitive to scale, (c) hard to interpret.
Also: (d) you want features for a classifier, and the two classes differ only along a direction of **small** variance. What happens?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

PCA is unsupervised: it never looks at the labels.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) Data on a Swiss roll or a circle: the intrinsic dimension is 1–2 but no linear subspace captures it; use Kernel PCA, Isomap, UMAP ([[Dimensionality Reduction]]).
(b) Income in kroner next to age in years: PC1 is essentially "income" (see Exercise 13).
(c) Each component mixes all original features with positive and negative weights, e.g. PC2 $=0.4\,$age$-0.3\,$height$+\dots$ has no obvious meaning (sparse PCA helps).
(d) PCA keeps high-variance directions and may throw away exactly the discriminative one; use a supervised method such as LDA.

</details>

## Part B · By hand

### Exercise 4 · Eigen-decomposition of a 2×2 covariance
*✍️ By hand · ★☆☆*

Let $S=\begin{pmatrix}2&1\\1&2\end{pmatrix}$. Find its eigenvalues, the first principal direction (unit length) and the
explained variance ratio of PC1.

In [ ]:
eigvals = None    # [lambda1, lambda2], descending
pc1 = None        # unit vector
evr1 = None

_S = np.array([[2., 1], [1, 2]]); _l, _V = np.linalg.eigh(_S)
check('eigenvalues', eigvals, _l[::-1])
check('PC1 (up to sign)', None if pc1 is None else np.abs(pc1), np.abs(_V[:, -1]))
check('explained variance ratio', evr1, _l[-1] / _l.sum())

<details>
<summary><b>💡 Hint</b></summary>

$\det(S-\lambda I)=(2-\lambda)^2-1=0$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$(2-\lambda)^2=1\Rightarrow\lambda\in\{3,1\}$. For $\lambda=3$: $(S-3I)v=0\Rightarrow v_1=v_2$, so $w_1=\tfrac1{\sqrt2}(1,1)^\top$.
Explained variance ratio $3/(3+1)=0.75$.

```python
eigvals = [3, 1]
pc1 = np.array([1, 1]) / np.sqrt(2)
evr1 = 0.75
```

</details>

### Exercise 5 · Project and reconstruct one point
*✍️ By hand · ★☆☆*

With $W=w_1=\tfrac1{\sqrt2}(1,1)^\top$ from the previous exercise, take the centred point $x=(2,0)$.
Compute its score $z=x^\top w_1$, its reconstruction $\hat x = z\,w_1$ and the squared reconstruction error $\lVert x-\hat x\rVert^2$.

In [ ]:
z = None
x_hat = None
err = None

_w = np.array([1., 1]) / np.sqrt(2); _x = np.array([2., 0])
check('score z', z, _x @ _w)
check('reconstruction', x_hat, (_x @ _w) * _w)
check('squared error', err, np.sum((_x - (_x @ _w) * _w) ** 2))

<details>
<summary><b>💡 Hint</b></summary>

$z = 2/\sqrt2=\sqrt2$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$z=\sqrt2\approx1.414$, $\hat x=\sqrt2\cdot\tfrac1{\sqrt2}(1,1)=(1,1)$, error $=\lVert(1,-1)\rVert^2=2$.
Check Pythagoras: $\lVert x\rVert^2=4=z^2+\text{err}=2+2$.

```python
z = np.sqrt(2)
x_hat = [1, 1]
err = 2.0
```

</details>

### Exercise 6 · From singular values to explained variance
*✍️ By hand · ★☆☆*

A centred data matrix with $N=9$ rows has singular values $\sigma=(6,3,1)$. Using $\lambda_i=\sigma_i^2/N$, give all eigenvalues of
$S$ and the explained variance ratio of PC1. How many components do you need for at least 95% explained variance?

In [ ]:
lams = None
evr_pc1 = None
k95 = None

_l = np.array([6., 3, 1]) ** 2 / 9
check('eigenvalues', lams, _l)
check('EVR of PC1', evr_pc1, _l[0] / _l.sum())
check('k for 95%', k95, int(np.searchsorted(np.cumsum(_l) / _l.sum(), 0.95) + 1))

<details>
<summary><b>💡 Hint</b></summary>

The ratio does not depend on $N$: it is $\sigma_1^2/\sum\sigma_i^2$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\lambda=(36,9,1)/9=(4,\,1,\,0.111)$. EVR of PC1 $=36/46\approx0.783$. Cumulative: $0.783,\ 0.978,\ 1$, so $k=2$.

```python
lams = [4, 1, 1/9]
evr_pc1 = 36/46
k95 = 2
```

</details>

### Exercise 7 · Deriving PCA with a Lagrange multiplier
*✍️ By hand · ★★☆*

Maximize the projected variance $w^\top S w$ subject to $w^\top w=1$. Show that the optimal $w$ is an eigenvector of $S$
and that the attained variance equals its eigenvalue. Why must $S$'s eigenvectors be orthogonal, so the next component is found the same way?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

$\mathcal L(w,\lambda)=w^\top S w-\lambda(w^\top w-1)$; use $\nabla_w(w^\top S w)=2Sw$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\nabla_w\mathcal L=2Sw-2\lambda w=0\Rightarrow Sw=\lambda w$: every stationary point is an eigenvector. Then $w^\top S w=\lambda w^\top w=\lambda$,
so the maximum is reached by the eigenvector of the **largest** eigenvalue.
$S$ is symmetric (and PSD), so by the spectral theorem it has an orthonormal eigenbasis $S=Q\Lambda Q^\top$. Maximizing again under the extra constraint $w\perp w_1$ gives the second eigenvector, and so on.

</details>

### Exercise 8 · Reconstruction error = discarded eigenvalues
*✍️ By hand · ★★☆*

Show that keeping the top $k$ eigenvectors gives mean squared reconstruction error $\frac1N\sum_i\lVert x_i-\hat x_i\rVert^2=\sum_{j>k}\lambda_j$.
Then compute it for $\lambda=(5,2,0.5,0.5)$ and $k=2$, and the retained variance fraction.

In [ ]:
mse_k2 = None
retained = None

_l = np.array([5, 2, .5, .5])
check('reconstruction MSE', mse_k2, _l[2:].sum())
check('retained fraction', retained, _l[:2].sum() / _l.sum())

<details>
<summary><b>💡 Hint</b></summary>

Write $x_i=\sum_j z_{ij}q_j$ in the full eigenbasis; the residual is $\sum_{j>k}z_{ij}q_j$ and $\frac1N\sum_i z_{ij}^2=\lambda_j$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$x_i-\hat x_i=\sum_{j>k}(q_j^\top x_i)q_j$, so by orthonormality $\lVert x_i-\hat x_i\rVert^2=\sum_{j>k}(q_j^\top x_i)^2$, and
$\frac1N\sum_i (q_j^\top x_i)^2=q_j^\top S q_j=\lambda_j$. Hence MSE $=\sum_{j>k}\lambda_j$.
Here: $0.5+0.5=1.0$; retained $7/8=0.875$.

```python
mse_k2 = 1.0
retained = 7/8
```

</details>

## Part C · Code from scratch

### Exercise 9 · PCA via the covariance eigendecomposition
*💻 Code · ★☆☆*

Implement `pca_eig(X, k)` following the note's algorithm: centre, $S=\frac1N X^\top X$, `np.linalg.eigh`, sort descending,
return `(W, lambdas, Z)` with $W$ the top-$k$ eigenvectors as columns, all eigenvalues sorted descending, and scores $Z=X_cW$.
Compare the explained variance ratio with `PCA` from scikit-learn on `X_c3`.

In [ ]:
def pca_eig(X, k):
    # TODO
    return None, None, None

W3, lam3, Z3 = pca_eig(X_c3, 2)
evr_mine = None if lam3 is None else lam3 / lam3.sum()

_p = PCA().fit(X_c3)
check('explained variance ratio', evr_mine, _p.explained_variance_ratio_)
check('components (up to sign)', None if W3 is None else np.abs(W3.T), np.abs(_p.components_[:2]))

<details>
<summary><b>💡 Hint</b></summary>

`eigh` returns eigenvalues in **ascending** order: reverse with `[::-1]`, and reorder the eigenvector columns the same way.

</details>

<details>
<summary><b>✅ Solution</b></summary>

scikit-learn divides by $N-1$ (`explained_variance_`), the note by $N$; the **ratios** and the directions are identical.
Eigenvectors are only defined up to sign, which is why we compare absolute values (sklearn fixes the sign with `svd_flip`).

```python
def pca_eig(X, k):
    Xc = X - X.mean(0)
    S = Xc.T @ Xc / len(X)
    lam, Q = np.linalg.eigh(S)
    order = np.argsort(lam)[::-1]
    lam, Q = lam[order], Q[:, order]
    W = Q[:, :k]
    return W, lam, Xc @ W

W3, lam3, Z3 = pca_eig(X_c3, 2)
evr_mine = lam3 / lam3.sum()
```

</details>

### Exercise 10 · PCA via the SVD
*💻 Code · ★☆☆*

Implement `pca_svd(X, k)` using `np.linalg.svd(Xc, full_matrices=False)`: components are the first $k$ rows of $V^\top$,
eigenvalues $\lambda_i=\sigma_i^2/N$, and scores $Z=U_k\Sigma_k$. Verify that the eigenvalues equal those from `pca_eig`.
Why is the SVD route preferred numerically?

In [ ]:
def pca_svd(X, k):
    # TODO: return W (d, k), lambdas (all), Z (N, k)
    return None, None, None

W3s, lam3s, Z3s = pca_svd(X_c3, 2)

_Xc = X_c3 - X_c3.mean(0)
check('eigenvalues = sigma^2 / N', lam3s, np.linalg.eigvalsh(_Xc.T @ _Xc / len(_Xc))[::-1])
check('scores U*Sigma = Xc W', None if Z3s is None else np.abs(Z3s), None if W3s is None else np.abs(_Xc @ W3s))

<details>
<summary><b>💡 Hint</b></summary>

$X=U\Sigma V^\top\Rightarrow X^\top X=V\Sigma^2V^\top$, so $V$ diagonalizes $S$ with eigenvalues $\sigma_i^2/N$. Also $XV=U\Sigma$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Forming $X^\top X$ squares the condition number, so small eigenvalues lose precision; the SVD works on $X$ directly. It also avoids the $d\times d$ matrix when $d\gg N$, and randomized/truncated SVD scales to large data (what sklearn uses).

```python
def pca_svd(X, k):
    Xc = X - X.mean(0)
    U, s, Vt = np.linalg.svd(Xc, full_matrices=False)
    return Vt[:k].T, s ** 2 / len(X), U[:, :k] * s[:k]

W3s, lam3s, Z3s = pca_svd(X_c3, 2)
```

</details>

### Exercise 11 · How many components for 95%?
*💻 Code · ★★☆*

On the 64-dimensional digits data (`X_digits`, not standardized) compute the cumulative explained variance ratio with your
`pca_svd`, plot it (scree / cumulative plot) and store the smallest $k$ reaching at least 95% in `k95_digits`.

In [ ]:
import matplotlib.pyplot as plt
k95_digits = None

check('k for 95% variance', k95_digits, PCA(0.95).fit(X_digits).n_components_)

<details>
<summary><b>💡 Hint</b></summary>

`np.cumsum(lam) / lam.sum()` then the first index where it is `>= 0.95` (remember +1 for a count).

</details>

<details>
<summary><b>✅ Solution</b></summary>

About 29 of 64 dimensions keep 95% of the variance: neighbouring pixels are strongly correlated and the border pixels are almost constant.
For visualization one would still use 2 components; for compression or as a pre-processing step 95% is a common rule of thumb.

```python
import matplotlib.pyplot as plt
_, lam_d, _ = pca_svd(X_digits, 2)
cum = np.cumsum(lam_d) / lam_d.sum()
plt.plot(np.arange(1, 65), cum, '.-'); plt.axhline(0.95, ls='--'); plt.xlabel('k'); plt.ylabel('cumulative EVR'); plt.show()
k95_digits = int(np.argmax(cum >= 0.95) + 1)
print(k95_digits)
```

</details>

### Exercise 12 · Reconstruction error curve
*💻 Code · ★★☆*

For $k\in\{1,2,5,10,20,40\}$ reconstruct the digits with $\hat X=ZW^\top+\bar x$ and store the mean squared reconstruction error
per sample ($\frac1N\sum_i\lVert x_i-\hat x_i\rVert^2$) in a list `recon_mse`. Confirm the identity from Exercise 8 numerically and
look at a reconstructed digit for $k=5$ and $k=20$.

In [ ]:
ks_r = [1, 2, 5, 10, 20, 40]
recon_mse = None

_Xc = X_digits - X_digits.mean(0); _l = np.linalg.eigvalsh(_Xc.T @ _Xc / len(_Xc))[::-1]
check('MSE equals sum of discarded eigenvalues', recon_mse, [_l[k:].sum() for k in ks_r], tol=1e-6)

<details>
<summary><b>💡 Hint</b></summary>

Don't forget to add the mean back after reconstructing from centred data.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The error drops quickly then flattens; at $k=20$ the reconstructed digits are clearly readable. The curve is exactly the tail sum of the eigenvalue spectrum.

```python
ks_r = [1, 2, 5, 10, 20, 40]
recon_mse = []
mu_d = X_digits.mean(0)
for k in ks_r:
    W, _, Z = pca_svd(X_digits, k)
    Xh = Z @ W.T + mu_d
    recon_mse.append(np.mean(np.sum((X_digits - Xh) ** 2, 1)))
print(np.round(recon_mse, 1))
```

</details>

### Exercise 13 · To standardize or not: the wine data
*💻 Code · ★★☆*

Load `load_wine()`. Run PCA on the raw features and on standardized features. For each, store the index of the feature with the
largest absolute loading in PC1 (`top_feature_raw`, `top_feature_std`) and the EVR of PC1 (`evr1_raw`, `evr1_std`).
Explain the difference.

In [ ]:
wine = load_wine()
Xw = wine.data
top_feature_raw = None
top_feature_std = None
evr1_raw = None
evr1_std = None

check('raw PC1 is the highest-variance feature', top_feature_raw, int(np.argmax(Xw.var(0))))
check('raw PC1 EVR is huge', None if evr1_raw is None else (evr1_raw > 0.99 and evr1_std < 0.5), True)
check('standardized PC1 is not dominated by it', None if top_feature_std is None else top_feature_std != top_feature_raw, True)

<details>
<summary><b>💡 Hint</b></summary>

Look at `Xw.var(0)` first: one feature (proline) has a variance about 1000 times larger than the others.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Raw: PC1 explains about 99.8% of the variance and is essentially the `proline` axis, simply because proline is measured in large numbers. That is a statement about units, not structure.
Standardized: PC1 explains about 36% and mixes many chemical measurements (flavanoids, phenols, …); this PCA reflects correlations, i.e. the correlation-matrix PCA.

```python
wine = load_wine()
Xw = wine.data
p_raw = PCA().fit(Xw)
p_std = PCA().fit(StandardScaler().fit_transform(Xw))
top_feature_raw = int(np.argmax(np.abs(p_raw.components_[0])))
top_feature_std = int(np.argmax(np.abs(p_std.components_[0])))
evr1_raw, evr1_std = p_raw.explained_variance_ratio_[0], p_std.explained_variance_ratio_[0]
print(wine.feature_names[top_feature_raw], evr1_raw, '|', wine.feature_names[top_feature_std], evr1_std)
```

</details>

### Exercise 14 · PCA whitening
*💻 Code · ★★☆*

Whitening maps centred data to $Z_w = X_c W\Lambda^{-1/2}$ (all components kept), so the new features are uncorrelated with unit variance.
Implement it for `X_c3` and verify that $\frac1N Z_w^\top Z_w = I$. Store the whitened data in `Zw`.

In [ ]:
Zw = None

check('covariance of whitened data is I', None if Zw is None else Zw.T @ Zw / len(Zw), np.eye(3))

<details>
<summary><b>💡 Hint</b></summary>

Use `pca_eig(X_c3, 3)` and divide each score column by $\sqrt{\lambda_j}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\frac1N Z_w^\top Z_w=\Lambda^{-1/2}W^\top S W\Lambda^{-1/2}=\Lambda^{-1/2}\Lambda\Lambda^{-1/2}=I$.
Whitening is used before ICA and in some image pipelines; beware it amplifies the low-variance (often noise) directions, so in practice add a small $\epsilon$ to $\lambda_j$.

```python
W_all, lam_all, Z_all = pca_eig(X_c3, 3)
Zw = Z_all / np.sqrt(lam_all)
```

</details>

### Exercise 15 · Power iteration for PC1
*💻 Code · ★★★*

Large-scale PCA rarely forms a full eigendecomposition. Implement `power_iteration(S, n_iter)`: start from a random unit vector,
repeat $v\leftarrow Sv/\lVert Sv\rVert$, return $(v, v^\top S v)$. Apply it to the digits covariance. Then get PC2 by **deflation**:
run it again on $S-\lambda_1v_1v_1^\top$. Store `v1, lam1, v2, lam2`.

In [ ]:
def power_iteration(S, n_iter=500, seed=0):
    # TODO
    return None, None

Xdc = X_digits - X_digits.mean(0)
S_digits = Xdc.T @ Xdc / len(Xdc)
v1, lam1 = power_iteration(S_digits)
v2, lam2 = None, None

_l, _V = np.linalg.eigh(S_digits)
check('|<v1, true PC1>| = 1', None if v1 is None else abs(v1 @ _V[:, -1]), 1.0, tol=1e-6)
check('lambda1', lam1, _l[-1])
check('lambda2 via deflation', lam2, _l[-2])

<details>
<summary><b>💡 Hint 1</b></summary>

Convergence rate is $|\lambda_2/\lambda_1|^t$: fast when the top eigenvalue is well separated.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Deflation removes the found direction: $S_2 = S-\lambda_1 v_1v_1^\top$ has the same eigenvectors, with $\lambda_1$ replaced by 0.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Writing $v_0=\sum_j c_jq_j$, after $t$ steps $S^tv_0=\sum_j c_j\lambda_j^tq_j$, dominated by $q_1$ when $\lambda_1>\lambda_2$.
Deflation then exposes $\lambda_2$. Errors accumulate with repeated deflation, so in practice one uses block methods (subspace/Lanczos iteration, randomized SVD).

```python
def power_iteration(S, n_iter=500, seed=0):
    v = np.random.default_rng(seed).normal(size=len(S))
    v /= np.linalg.norm(v)
    for _ in range(n_iter):
        v = S @ v
        v /= np.linalg.norm(v)
    return v, v @ S @ v

Xdc = X_digits - X_digits.mean(0)
S_digits = Xdc.T @ Xdc / len(Xdc)
v1, lam1 = power_iteration(S_digits)
v2, lam2 = power_iteration(S_digits - lam1 * np.outer(v1, v1))
```

</details>

### Exercise 16 · Probabilistic PCA in closed form
*💻 Code · ★★★*

PPCA ([[Probabilistic Graphical Models]]) assumes $x=Wz+\mu+\varepsilon$, $z\sim\mathcal N(0,I_k)$, $\varepsilon\sim\mathcal N(0,\sigma^2I)$, so
$x\sim\mathcal N(\mu, WW^\top+\sigma^2I)$. Tipping & Bishop's MLE is
$\sigma^2_{ML}=\frac1{d-k}\sum_{j>k}\lambda_j$ and $W_{ML}=U_k(\Lambda_k-\sigma^2I)^{1/2}$ (eigen-pairs of the sample covariance).

Compute both for the digits with $k=10$ using `np.cov` (divides by $N-1$, like sklearn) and store `sigma2_ppca` and the
model covariance `C_ppca` $=W W^\top+\sigma^2I$.

In [ ]:
sigma2_ppca = None
C_ppca = None

_p = PCA(10).fit(X_digits)
check('sigma^2 (sklearn noise_variance_)', sigma2_ppca, _p.noise_variance_)
check('model covariance (sklearn get_covariance)', C_ppca, _p.get_covariance(), tol=1e-6)

<details>
<summary><b>💡 Hint</b></summary>

`lam, U = np.linalg.eigh(np.cov(X_digits, rowvar=False))`, reverse the order, then plug in.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The noise variance is the average variance "left over" in the discarded directions; the retained directions get variance $\lambda_j$ exactly ($\lambda_j-\sigma^2$ from $W$ plus $\sigma^2$ noise).
As $\sigma^2\to0$, PPCA recovers ordinary PCA. Being a proper density, PPCA gives a likelihood (sklearn's `PCA.score`), can handle missing data via EM, and is the linear-Gaussian ancestor of factor analysis and VAEs.

```python
lam_c, U_c = np.linalg.eigh(np.cov(X_digits, rowvar=False))
lam_c, U_c = lam_c[::-1], U_c[:, ::-1]
k, d = 10, X_digits.shape[1]
sigma2_ppca = lam_c[k:].mean()
W_ml = U_c[:, :k] * np.sqrt(lam_c[:k] - sigma2_ppca)
C_ppca = W_ml @ W_ml.T + sigma2_ppca * np.eye(d)
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [PCA](PCA.md).*